# ticker_attention_diagnostics

Diagnostics for the thesis question: do some tickers have too little Google, Reddit, or GDELT attention for alternative-data features to be useful?

The notebook first measures raw attention coverage by ticker, then compares compact logistic-regression models against a price+volume baseline at ticker level. It reports both pooled-model per-ticker lifts and ticker-specific model lifts.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [ ]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.split_utils import make_split_dates, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

MIN_TICKER_TRAIN_ROWS = 300
MIN_TICKER_VALIDATION_ROWS = 80
MIN_TICKER_TEST_ROWS = 80

LOGREG_PARAM_GRID = [
    {
        "param_set": "logreg_l2_C0p1_balanced",
        "penalty": "l2",
        "C": 0.1,
        "solver": "lbfgs",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l2_C1_balanced",
        "penalty": "l2",
        "C": 1.0,
        "solver": "lbfgs",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l1_C0p1_balanced",
        "penalty": "l1",
        "C": 0.1,
        "solver": "liblinear",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l2_C0p1_unweighted",
        "penalty": "l2",
        "C": 0.1,
        "solver": "lbfgs",
        "class_weight": None,
        "max_iter": 3000,
    },
]

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SETS = feature_grid.feature_sets
FEATURE_FAMILIES = {
    feature_set: metadata["feature_family"]
    for feature_set, metadata in feature_grid.feature_set_metadata.items()
}
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
ATTENTION_FEATURE_SETS = [
    feature_set for feature_set in FEATURE_SETS_TO_TEST if "attention" in FEATURE_FAMILIES[feature_set]
]

pd.DataFrame(LOGREG_PARAM_GRID)


In [ ]:
from __future__ import annotations

raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {"train": train_dates, "validation": validation_dates, "test": test_dates}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)

split_summary_df = (
    modeled_df[modeled_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        modeled_rows=("target", "size"),
        dates=("date", "nunique"),
        positive_rate=("target", "mean"),
        date_min=("date", "min"),
        date_max=("date", "max"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

ticker_split_summary_df = (
    modeled_df[modeled_df["split"].isin(["train", "validation", "test"])]
    .groupby(["ticker", "split"])
    .agg(
        modeled_rows=("target", "size"),
        dates=("date", "nunique"),
        positive_rate=("target", "mean"),
    )
    .reset_index()
)

split_summary_df


In [ ]:
from __future__ import annotations

ATTENTION_SIGNAL_COLUMNS = {
    "google": "google_trends_score",
    "reddit": "comm_reddit_posts",
    "gdelt": "gdelt_articles",
}


def attention_coverage_summary(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for group_key, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = dict(zip(group_cols, group_key))
        row["rows"] = len(group)
        for signal_name, column in ATTENTION_SIGNAL_COLUMNS.items():
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{signal_name}_available_rate"] = float(values.notna().mean())
            row[f"{signal_name}_nonzero_rate"] = float(values.fillna(0.0).gt(0.0).mean())
            row[f"{signal_name}_median"] = float(values.median())
            row[f"{signal_name}_mean"] = float(values.mean())
            row[f"{signal_name}_p90"] = float(values.quantile(0.90))
            row[f"{signal_name}_total"] = float(values.fillna(0.0).sum())
        rows.append(row)
    return pd.DataFrame(rows)


attention_coverage_by_ticker_df = attention_coverage_summary(feature_df, ["ticker"])

for signal_name in ATTENTION_SIGNAL_COLUMNS:
    attention_coverage_by_ticker_df[f"{signal_name}_rank"] = attention_coverage_by_ticker_df[
        f"{signal_name}_mean"
    ].rank(ascending=False, method="min")

attention_coverage_by_ticker_df["avg_attention_rank"] = attention_coverage_by_ticker_df[
    ["google_rank", "reddit_rank", "gdelt_rank"]
].mean(axis=1)
attention_coverage_by_ticker_df["low_overall_attention_flag"] = (
    attention_coverage_by_ticker_df["avg_attention_rank"]
    >= attention_coverage_by_ticker_df["avg_attention_rank"].quantile(0.75)
)
attention_coverage_by_ticker_df["weakest_attention_source"] = attention_coverage_by_ticker_df[
    ["google_rank", "reddit_rank", "gdelt_rank"]
].idxmax(axis=1).str.replace("_rank", "", regex=False)

total_columns = [f"{signal_name}_total" for signal_name in ATTENTION_SIGNAL_COLUMNS]
for total_column in total_columns:
    attention_coverage_by_ticker_df[total_column.replace("_total", "_share")] = (
        attention_coverage_by_ticker_df[total_column] / attention_coverage_by_ticker_df[total_column].sum()
    )

attention_coverage_by_ticker_split_df = attention_coverage_summary(
    feature_df[feature_df["split"].isin(["train", "validation", "test"])],
    ["ticker", "split"],
)

attention_coverage_display_columns = [
    "ticker",
    "rows",
    "google_mean",
    "reddit_mean",
    "gdelt_mean",
    "google_rank",
    "reddit_rank",
    "gdelt_rank",
    "avg_attention_rank",
    "weakest_attention_source",
    "low_overall_attention_flag",
]

attention_coverage_by_ticker_df[attention_coverage_display_columns].sort_values(
    ["low_overall_attention_flag", "avg_attention_rank"],
    ascending=[False, False],
).reset_index(drop=True)


In [ ]:
from __future__ import annotations

missing_feature_columns = sorted(
    {
        feature
        for feature_set in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[feature_set]
        if feature not in feature_df.columns
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "feature_family": FEATURE_FAMILIES[feature_set],
            "n_features": len(features),
            "is_attention_set": feature_set in ATTENTION_FEATURE_SETS,
            "features": features,
        }
        for feature_set, features in FEATURE_SETS.items()
    ]
)

print(f"Tickers: {modeled_df['ticker'].nunique()}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {len(ATTENTION_FEATURE_SETS)}")
print(f"Logistic-regression parameter sets: {len(LOGREG_PARAM_GRID)}")
print(
    "Ticker-specific fits: "
    f"{modeled_df['ticker'].nunique() * len(FEATURE_SETS_TO_TEST) * len(LOGREG_PARAM_GRID)}"
)

candidate_feature_sets_df


In [ ]:
from __future__ import annotations

LOGREG_PARAM_COLUMNS = ["penalty", "C", "solver", "class_weight", "max_iter"]


def build_logreg_pipeline_from_params(params: dict) -> Pipeline:
    model_params = dict(params)
    model_params.pop("param_set", None)
    model_params.setdefault("random_state", CONFIG["random_state"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(**model_params)),
        ]
    )


def best_threshold_for_balanced_accuracy(
    y_true: pd.Series,
    scores: np.ndarray,
    *,
    default_threshold: float = 0.0,
) -> tuple[float, float]:
    return ClassificationMetrics.best_threshold_for_balanced_accuracy(
        y_true,
        scores,
        min_quantile=CONFIG["threshold_min_quantile"],
        max_quantile=CONFIG["threshold_max_quantile"],
        grid_size=CONFIG["threshold_grid_size"],
        default_threshold=default_threshold,
    )


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    return ClassificationMetrics.metrics_from_scores(y_true, scores, threshold)


def add_param_columns(row: dict, params: dict) -> None:
    for column in LOGREG_PARAM_COLUMNS:
        row[column] = params.get(column)


def evaluate_logreg_params(
    *,
    feature_set_name: str,
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    scope: str,
    ticker: str | None = None,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    features = FEATURE_SETS[feature_set_name]
    pipeline = build_logreg_pipeline_from_params(params)
    pipeline.fit(train_input_df[features], train_input_df["target"])
    scores = pipeline.decision_function(eval_input_df[features])

    threshold_source = "fixed"
    if tune_threshold:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(
            eval_input_df["target"],
            scores,
            default_threshold=0.0,
        )
        threshold_source = "validation_tuned"
    else:
        threshold_selection_balanced_accuracy = np.nan
        if decision_threshold is None:
            decision_threshold = 0.0
            threshold_source = "default_zero"

    metric_result = metrics_from_scores(eval_input_df["target"], scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    row = {
        "scope": scope,
        "ticker": ticker if ticker is not None else "ALL",
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": FEATURE_FAMILIES[feature_set_name],
        "is_attention_set": feature_set_name in ATTENTION_FEATURE_SETS,
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "n_features": len(features),
        "train_rows": len(train_input_df),
        "eval_rows": len(eval_input_df),
        "eval_positive_rate": float(eval_input_df["target"].mean()),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        **metric_result,
    }
    add_param_columns(row, params)

    if not return_predictions:
        return row

    predictions_df = eval_input_df[["date", "ticker", "target"]].copy()
    predictions_df["scope"] = scope
    predictions_df["feature_set"] = feature_set_name
    predictions_df["feature_family"] = FEATURE_FAMILIES[feature_set_name]
    predictions_df["param_set"] = params["param_set"]
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def select_params_on_validation(
    *,
    feature_set_name: str,
    train_input_df: pd.DataFrame,
    validation_input_df: pd.DataFrame,
    scope: str,
    ticker: str | None = None,
) -> tuple[dict, pd.DataFrame]:
    validation_rows = []
    for params in LOGREG_PARAM_GRID:
        validation_rows.append(
            evaluate_logreg_params(
                feature_set_name=feature_set_name,
                params=params,
                train_input_df=train_input_df,
                eval_input_df=validation_input_df,
                split_name="validation",
                scope=scope,
                ticker=ticker,
                tune_threshold=True,
            )
        )
    validation_results_df = pd.DataFrame(validation_rows)
    best_row = (
        validation_results_df.sort_values(
            ["balanced_accuracy", "f1_score", "accuracy", "param_set"],
            ascending=[False, False, False, True],
        )
        .head(1)
        .iloc[0]
        .to_dict()
    )
    return best_row, validation_results_df


def metrics_by_ticker_from_predictions(predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker, ticker_predictions_df in predictions_df.groupby("ticker"):
        metric_result = metrics_from_scores(
            ticker_predictions_df["target"],
            ticker_predictions_df["score"].to_numpy(),
            float(ticker_predictions_df["decision_threshold"].iloc[0]),
        )
        metric_result.pop("preds")
        rows.append(
            {
                "ticker": ticker,
                "feature_set": ticker_predictions_df["feature_set"].iloc[0],
                "feature_family": ticker_predictions_df["feature_family"].iloc[0],
                "is_attention_set": ticker_predictions_df["feature_set"].iloc[0] in ATTENTION_FEATURE_SETS,
                "param_set": ticker_predictions_df["param_set"].iloc[0],
                "test_rows": len(ticker_predictions_df),
                **metric_result,
            }
        )
    return pd.DataFrame(rows)


In [ ]:
from __future__ import annotations

pooled_validation_rows = []
pooled_test_rows = []
pooled_prediction_frames = []
pooled_per_ticker_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    best_validation_row, validation_results_df = select_params_on_validation(
        feature_set_name=feature_set_name,
        train_input_df=train_df,
        validation_input_df=validation_df,
        scope="pooled",
    )
    pooled_validation_rows.extend(validation_results_df.to_dict(orient="records"))
    selected_params = next(
        params for params in LOGREG_PARAM_GRID if params["param_set"] == best_validation_row["param_set"]
    )
    test_row, predictions_df = evaluate_logreg_params(
        feature_set_name=feature_set_name,
        params=selected_params,
        train_input_df=train_df,
        eval_input_df=test_df,
        split_name="test",
        scope="pooled",
        decision_threshold=best_validation_row["decision_threshold"],
        return_predictions=True,
    )
    test_row["validation_balanced_accuracy"] = best_validation_row["balanced_accuracy"]
    test_row["validation_f1_score"] = best_validation_row["f1_score"]
    pooled_test_rows.append(test_row)
    pooled_prediction_frames.append(predictions_df)
    pooled_per_ticker_rows.append(metrics_by_ticker_from_predictions(predictions_df))

pooled_validation_results_df = pd.DataFrame(pooled_validation_rows)
pooled_feature_set_test_results_df = pd.DataFrame(pooled_test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
pooled_test_predictions_df = pd.concat(pooled_prediction_frames, ignore_index=True)
pooled_per_ticker_test_results_df = pd.concat(pooled_per_ticker_rows, ignore_index=True)

pooled_baseline_by_ticker_df = pooled_per_ticker_test_results_df[
    pooled_per_ticker_test_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["ticker", "accuracy", "balanced_accuracy", "f1_score"]].rename(
    columns={
        "accuracy": "baseline_accuracy",
        "balanced_accuracy": "baseline_balanced_accuracy",
        "f1_score": "baseline_f1_score",
    }
)

pooled_per_ticker_lift_vs_price_volume_df = pooled_per_ticker_test_results_df.merge(
    pooled_baseline_by_ticker_df,
    on="ticker",
    how="left",
)
pooled_per_ticker_lift_vs_price_volume_df["accuracy_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["accuracy"]
    - pooled_per_ticker_lift_vs_price_volume_df["baseline_accuracy"]
)
pooled_per_ticker_lift_vs_price_volume_df["balanced_accuracy_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["balanced_accuracy"]
    - pooled_per_ticker_lift_vs_price_volume_df["baseline_balanced_accuracy"]
)
pooled_per_ticker_lift_vs_price_volume_df["f1_score_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["f1_score"] - pooled_per_ticker_lift_vs_price_volume_df["baseline_f1_score"]
)

pooled_feature_set_test_results_df


In [ ]:
from __future__ import annotations

ticker_specific_validation_rows = []
ticker_specific_test_rows = []
skipped_ticker_models = []

for ticker in sorted(modeled_df["ticker"].unique()):
    ticker_train_df = train_df[train_df["ticker"].eq(ticker)].copy()
    ticker_validation_df = validation_df[validation_df["ticker"].eq(ticker)].copy()
    ticker_test_df = test_df[test_df["ticker"].eq(ticker)].copy()

    if (
        len(ticker_train_df) < MIN_TICKER_TRAIN_ROWS
        or len(ticker_validation_df) < MIN_TICKER_VALIDATION_ROWS
        or len(ticker_test_df) < MIN_TICKER_TEST_ROWS
    ):
        skipped_ticker_models.append(
            {
                "ticker": ticker,
                "train_rows": len(ticker_train_df),
                "validation_rows": len(ticker_validation_df),
                "test_rows": len(ticker_test_df),
            }
        )
        continue

    for feature_set_name in FEATURE_SETS_TO_TEST:
        best_validation_row, validation_results_df = select_params_on_validation(
            feature_set_name=feature_set_name,
            train_input_df=ticker_train_df,
            validation_input_df=ticker_validation_df,
            scope="ticker_specific",
            ticker=ticker,
        )
        ticker_specific_validation_rows.extend(validation_results_df.to_dict(orient="records"))
        selected_params = next(
            params for params in LOGREG_PARAM_GRID if params["param_set"] == best_validation_row["param_set"]
        )
        test_row = evaluate_logreg_params(
            feature_set_name=feature_set_name,
            params=selected_params,
            train_input_df=ticker_train_df,
            eval_input_df=ticker_test_df,
            split_name="test",
            scope="ticker_specific",
            ticker=ticker,
            decision_threshold=best_validation_row["decision_threshold"],
        )
        test_row["validation_balanced_accuracy"] = best_validation_row["balanced_accuracy"]
        test_row["validation_f1_score"] = best_validation_row["f1_score"]
        ticker_specific_test_rows.append(test_row)

ticker_specific_validation_results_df = pd.DataFrame(ticker_specific_validation_rows)
ticker_specific_best_models_df = pd.DataFrame(ticker_specific_test_rows).sort_values(
    ["ticker", "balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)
skipped_ticker_models_df = pd.DataFrame(skipped_ticker_models)

ticker_baseline_df = ticker_specific_best_models_df[
    ticker_specific_best_models_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["ticker", "accuracy", "balanced_accuracy", "f1_score"]].rename(
    columns={
        "accuracy": "baseline_accuracy",
        "balanced_accuracy": "baseline_balanced_accuracy",
        "f1_score": "baseline_f1_score",
    }
)

ticker_specific_lift_vs_price_volume_df = ticker_specific_best_models_df.merge(
    ticker_baseline_df,
    on="ticker",
    how="left",
)
ticker_specific_lift_vs_price_volume_df["accuracy_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["accuracy"]
    - ticker_specific_lift_vs_price_volume_df["baseline_accuracy"]
)
ticker_specific_lift_vs_price_volume_df["balanced_accuracy_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["balanced_accuracy"]
    - ticker_specific_lift_vs_price_volume_df["baseline_balanced_accuracy"]
)
ticker_specific_lift_vs_price_volume_df["f1_score_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["f1_score"] - ticker_specific_lift_vs_price_volume_df["baseline_f1_score"]
)

ticker_specific_best_models_df.head(20)


In [ ]:
from __future__ import annotations

def best_lift_rows(
    frame: pd.DataFrame,
    *,
    feature_filter: pd.Series,
    prefix: str,
) -> pd.DataFrame:
    best = (
        frame[feature_filter]
        .sort_values(
            ["ticker", "balanced_accuracy_lift_vs_price_volume", "f1_score_lift_vs_price_volume", "accuracy_lift_vs_price_volume"],
            ascending=[True, False, False, False],
        )
        .groupby("ticker", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return best[
        [
            "ticker",
            "feature_set",
            "feature_family",
            "balanced_accuracy_lift_vs_price_volume",
            "accuracy_lift_vs_price_volume",
            "f1_score_lift_vs_price_volume",
            "balanced_accuracy",
            "accuracy",
            "f1_score",
        ]
    ].rename(
        columns={
            "feature_set": f"{prefix}_best_feature_set",
            "feature_family": f"{prefix}_best_feature_family",
            "balanced_accuracy_lift_vs_price_volume": f"{prefix}_balanced_accuracy_lift",
            "accuracy_lift_vs_price_volume": f"{prefix}_accuracy_lift",
            "f1_score_lift_vs_price_volume": f"{prefix}_f1_score_lift",
            "balanced_accuracy": f"{prefix}_balanced_accuracy",
            "accuracy": f"{prefix}_accuracy",
            "f1_score": f"{prefix}_f1_score",
        }
    )


pooled_best_attention_lift_df = best_lift_rows(
    pooled_per_ticker_lift_vs_price_volume_df,
    feature_filter=pooled_per_ticker_lift_vs_price_volume_df["is_attention_set"],
    prefix="pooled_attention",
)
pooled_best_any_alternative_lift_df = best_lift_rows(
    pooled_per_ticker_lift_vs_price_volume_df,
    feature_filter=~pooled_per_ticker_lift_vs_price_volume_df["feature_family"].isin(["price only", "price + volume"]),
    prefix="pooled_any_alternative",
)
ticker_best_attention_lift_df = best_lift_rows(
    ticker_specific_lift_vs_price_volume_df,
    feature_filter=ticker_specific_lift_vs_price_volume_df["is_attention_set"],
    prefix="ticker_attention",
)
ticker_best_any_alternative_lift_df = best_lift_rows(
    ticker_specific_lift_vs_price_volume_df,
    feature_filter=~ticker_specific_lift_vs_price_volume_df["feature_family"].isin(["price only", "price + volume"]),
    prefix="ticker_any_alternative",
)

attention_coverage_model_lift_df = (
    attention_coverage_by_ticker_df[attention_coverage_display_columns]
    .merge(pooled_best_attention_lift_df, on="ticker", how="left")
    .merge(pooled_best_any_alternative_lift_df, on="ticker", how="left")
    .merge(ticker_best_attention_lift_df, on="ticker", how="left")
    .merge(ticker_best_any_alternative_lift_df, on="ticker", how="left")
    .sort_values(["low_overall_attention_flag", "avg_attention_rank"], ascending=[False, False])
    .reset_index(drop=True)
)

attention_coverage_model_lift_df


In [ ]:
from __future__ import annotations

correlation_metric_pairs = [
    ("google_mean", "Google mean attention"),
    ("reddit_mean", "Reddit mean attention"),
    ("gdelt_mean", "GDELT mean attention"),
    ("avg_attention_rank", "Average attention rank; higher means weaker attention"),
]
lift_metric_pairs = [
    ("pooled_attention_balanced_accuracy_lift", "Pooled model attention lift"),
    ("pooled_any_alternative_balanced_accuracy_lift", "Pooled model any-alt lift"),
    ("ticker_attention_balanced_accuracy_lift", "Ticker-specific attention lift"),
    ("ticker_any_alternative_balanced_accuracy_lift", "Ticker-specific any-alt lift"),
]

correlation_rows = []
for coverage_column, coverage_label in correlation_metric_pairs:
    for lift_column, lift_label in lift_metric_pairs:
        valid_df = attention_coverage_model_lift_df[[coverage_column, lift_column]].dropna()
        correlation_rows.append(
            {
                "coverage_metric": coverage_label,
                "lift_metric": lift_label,
                "n_tickers": len(valid_df),
                "pearson_correlation": valid_df[coverage_column].corr(valid_df[lift_column], method="pearson"),
                "spearman_correlation": valid_df[coverage_column].corr(valid_df[lift_column], method="spearman"),
            }
        )

coverage_lift_correlation_df = pd.DataFrame(correlation_rows)

coverage_lift_correlation_df


In [ ]:
from __future__ import annotations

def concern_label(row: pd.Series) -> str:
    lift = row["ticker_attention_balanced_accuracy_lift"]
    if pd.isna(lift):
        return "not enough data for ticker-specific conclusion"
    if row["low_overall_attention_flag"] and lift <= 0:
        return "supports concern: low attention and no attention lift"
    if row["low_overall_attention_flag"] and lift > 0:
        return "low attention but attention still helped"
    if not row["low_overall_attention_flag"] and lift <= 0:
        return "attention available but no attention lift"
    return "attention available and attention helped"


concern_summary_columns = [
    "ticker",
    "low_overall_attention_flag",
    "weakest_attention_source",
    "google_mean",
    "reddit_mean",
    "gdelt_mean",
    "avg_attention_rank",
    "ticker_attention_best_feature_set",
    "ticker_attention_balanced_accuracy_lift",
    "ticker_any_alternative_best_feature_set",
    "ticker_any_alternative_balanced_accuracy_lift",
    "pooled_attention_best_feature_set",
    "pooled_attention_balanced_accuracy_lift",
]

concern_summary_df = attention_coverage_model_lift_df[concern_summary_columns].copy()
concern_summary_df["diagnostic_label"] = concern_summary_df.apply(concern_label, axis=1)
concern_summary_df = concern_summary_df.sort_values(
    ["low_overall_attention_flag", "ticker_attention_balanced_accuracy_lift"],
    ascending=[False, True],
).reset_index(drop=True)

concern_summary_df


## Basic Model Family Check

This section uses a deliberately small set of basic models to verify whether alternative-data feature sets improve test balanced accuracy versus `price + volume`. These are diagnostics, not a broad tuning grid.


In [ ]:
from __future__ import annotations

BASIC_MODEL_CONFIGS = [
    {
        "model_name": "dummy_most_frequent",
        "model_family": "dummy",
        "default_threshold": 0.5,
        "tune_threshold": False,
    },
    {
        "model_name": "logreg_l2_C0p1_balanced",
        "model_family": "logistic_regression",
        "default_threshold": 0.0,
        "tune_threshold": True,
        "model_params": {
            "penalty": "l2",
            "C": 0.1,
            "solver": "lbfgs",
            "class_weight": "balanced",
            "max_iter": 3000,
            "random_state": CONFIG["random_state"],
        },
    },
    {
        "model_name": "linear_svm_C0p1_balanced",
        "model_family": "linear_svm",
        "default_threshold": 0.0,
        "tune_threshold": True,
        "model_params": {
            "C": 0.1,
            "class_weight": "balanced",
            "dual": False,
            "max_iter": 5000,
            "random_state": CONFIG["random_state"],
        },
    },
    {
        "model_name": "rf_shallow_balanced",
        "model_family": "random_forest",
        "default_threshold": 0.5,
        "tune_threshold": True,
        "model_params": {
            "n_estimators": 300,
            "max_depth": 4,
            "min_samples_leaf": 25,
            "min_samples_split": 50,
            "max_features": "sqrt",
            "class_weight": "balanced_subsample",
            "bootstrap": True,
            "n_jobs": -1,
            "random_state": CONFIG["random_state"],
        },
    },
]


def build_basic_model_pipeline(model_config: dict) -> Pipeline:
    model_family = model_config["model_family"]
    if model_family == "dummy":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        )
    if model_family == "logistic_regression":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(**model_config["model_params"])),
            ]
        )
    if model_family == "linear_svm":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LinearSVC(**model_config["model_params"])),
            ]
        )
    if model_family == "random_forest":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(**model_config["model_params"])),
            ]
        )
    raise ValueError(f"Unsupported model family: {model_family}")


def score_basic_model_pipeline(pipeline: Pipeline, features: list[str], frame: pd.DataFrame) -> np.ndarray:
    x_values = frame[features]
    if hasattr(pipeline, "decision_function"):
        return np.asarray(pipeline.decision_function(x_values), dtype=float)
    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(x_values)
        if probabilities.shape[1] == 1:
            predicted_class = pipeline.classes_[0]
            return np.ones(len(frame), dtype=float) if predicted_class == 1 else np.zeros(len(frame), dtype=float)
        positive_class_index = list(pipeline.classes_).index(1)
        return np.asarray(probabilities[:, positive_class_index], dtype=float)
    return np.asarray(pipeline.predict(x_values), dtype=float)


def evaluate_basic_model_feature_set(
    *,
    model_config: dict,
    feature_set_name: str,
    train_input_df: pd.DataFrame,
    validation_input_df: pd.DataFrame,
    test_input_df: pd.DataFrame,
) -> tuple[dict, pd.DataFrame]:
    features = FEATURE_SETS[feature_set_name]
    pipeline = build_basic_model_pipeline(model_config)
    pipeline.fit(train_input_df[features], train_input_df["target"])

    validation_scores = score_basic_model_pipeline(pipeline, features, validation_input_df)
    if model_config["tune_threshold"]:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(
            validation_input_df["target"],
            validation_scores,
            default_threshold=float(model_config["default_threshold"]),
        )
        threshold_source = "validation_tuned"
    else:
        decision_threshold = float(model_config["default_threshold"])
        threshold_selection_balanced_accuracy = np.nan
        threshold_source = "default"

    validation_metrics = metrics_from_scores(validation_input_df["target"], validation_scores, float(decision_threshold))
    validation_metrics.pop("preds")

    test_scores = score_basic_model_pipeline(pipeline, features, test_input_df)
    test_metrics = metrics_from_scores(test_input_df["target"], test_scores, float(decision_threshold))
    test_predictions = test_metrics.pop("preds")

    row = {
        "model_name": model_config["model_name"],
        "model_family": model_config["model_family"],
        "feature_set": feature_set_name,
        "feature_family": FEATURE_FAMILIES[feature_set_name],
        "is_attention_set": feature_set_name in ATTENTION_FEATURE_SETS,
        "n_features": len(features),
        "features": ", ".join(features),
        "train_rows": len(train_input_df),
        "validation_rows": len(validation_input_df),
        "test_rows": len(test_input_df),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
        "validation_f1_score": validation_metrics["f1_score"],
        "test_accuracy": test_metrics["accuracy"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_f1_score": test_metrics["f1_score"],
    }

    predictions_df = test_input_df[["date", "ticker", "target"]].copy()
    predictions_df["model_name"] = model_config["model_name"]
    predictions_df["model_family"] = model_config["model_family"]
    predictions_df["feature_set"] = feature_set_name
    predictions_df["feature_family"] = FEATURE_FAMILIES[feature_set_name]
    predictions_df["is_attention_set"] = feature_set_name in ATTENTION_FEATURE_SETS
    predictions_df["score"] = test_scores
    predictions_df["prediction"] = test_predictions
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


basic_model_rows = []
basic_model_prediction_frames = []

for model_config in BASIC_MODEL_CONFIGS:
    for feature_set_name in FEATURE_SETS_TO_TEST:
        result_row, predictions_df = evaluate_basic_model_feature_set(
            model_config=model_config,
            feature_set_name=feature_set_name,
            train_input_df=train_df,
            validation_input_df=validation_df,
            test_input_df=test_df,
        )
        basic_model_rows.append(result_row)
        basic_model_prediction_frames.append(predictions_df)

basic_model_feature_set_test_results_df = pd.DataFrame(basic_model_rows).sort_values(
    ["model_name", "test_balanced_accuracy", "test_f1_score", "test_accuracy", "feature_set"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)
basic_model_test_predictions_df = pd.concat(basic_model_prediction_frames, ignore_index=True)

basic_model_feature_set_test_results_df


In [ ]:
from __future__ import annotations

basic_model_baseline_df = basic_model_feature_set_test_results_df[
    basic_model_feature_set_test_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][
    [
        "model_name",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_f1_score",
        "validation_balanced_accuracy",
        "validation_f1_score",
    ]
].rename(
    columns={
        "test_accuracy": "baseline_test_accuracy",
        "test_balanced_accuracy": "baseline_test_balanced_accuracy",
        "test_f1_score": "baseline_test_f1_score",
        "validation_balanced_accuracy": "baseline_validation_balanced_accuracy",
        "validation_f1_score": "baseline_validation_f1_score",
    }
)

basic_model_alternative_lift_df = basic_model_feature_set_test_results_df.merge(
    basic_model_baseline_df,
    on="model_name",
    how="left",
)
basic_model_alternative_lift_df["test_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_accuracy"] - basic_model_alternative_lift_df["baseline_test_accuracy"]
)
basic_model_alternative_lift_df["test_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_balanced_accuracy"]
    - basic_model_alternative_lift_df["baseline_test_balanced_accuracy"]
)
basic_model_alternative_lift_df["test_f1_score_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_f1_score"] - basic_model_alternative_lift_df["baseline_test_f1_score"]
)
basic_model_alternative_lift_df["validation_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["validation_balanced_accuracy"]
    - basic_model_alternative_lift_df["baseline_validation_balanced_accuracy"]
)

basic_model_validation_selected_alternative_df = (
    basic_model_alternative_lift_df[
        ~basic_model_alternative_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "validation_balanced_accuracy", "validation_f1_score", "test_balanced_accuracy"],
        ascending=[True, False, False, False],
    )
    .groupby("model_name", as_index=False)
    .head(1)
    .sort_values("test_balanced_accuracy_lift_vs_price_volume", ascending=False)
    .reset_index(drop=True)
)

basic_model_best_test_alternative_df = (
    basic_model_alternative_lift_df[
        ~basic_model_alternative_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "test_balanced_accuracy_lift_vs_price_volume", "test_f1_score_lift_vs_price_volume"],
        ascending=[True, False, False],
    )
    .groupby("model_name", as_index=False)
    .head(1)
    .sort_values("test_balanced_accuracy_lift_vs_price_volume", ascending=False)
    .reset_index(drop=True)
)

basic_model_alternative_data_question_df = basic_model_validation_selected_alternative_df[
    [
        "model_name",
        "model_family",
        "feature_set",
        "feature_family",
        "is_attention_set",
        "validation_balanced_accuracy",
        "baseline_validation_balanced_accuracy",
        "validation_balanced_accuracy_lift_vs_price_volume",
        "test_balanced_accuracy",
        "baseline_test_balanced_accuracy",
        "test_balanced_accuracy_lift_vs_price_volume",
        "test_accuracy",
        "baseline_test_accuracy",
        "test_accuracy_lift_vs_price_volume",
        "test_f1_score",
        "baseline_test_f1_score",
        "test_f1_score_lift_vs_price_volume",
    ]
].copy()
basic_model_alternative_data_question_df["helps_test_balanced_accuracy"] = (
    basic_model_alternative_data_question_df["test_balanced_accuracy_lift_vs_price_volume"] > 0
)


def basic_metrics_by_ticker(predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_name, feature_set, ticker), group in predictions_df.groupby(["model_name", "feature_set", "ticker"]):
        metric_result = metrics_from_scores(
            group["target"],
            group["score"].to_numpy(),
            float(group["decision_threshold"].iloc[0]),
        )
        metric_result.pop("preds")
        rows.append(
            {
                "model_name": model_name,
                "model_family": group["model_family"].iloc[0],
                "ticker": ticker,
                "feature_set": feature_set,
                "feature_family": group["feature_family"].iloc[0],
                "is_attention_set": bool(group["is_attention_set"].iloc[0]),
                "test_rows": len(group),
                "test_accuracy": metric_result["accuracy"],
                "test_balanced_accuracy": metric_result["balanced_accuracy"],
                "test_f1_score": metric_result["f1_score"],
            }
        )
    return pd.DataFrame(rows)


basic_model_per_ticker_results_df = basic_metrics_by_ticker(basic_model_test_predictions_df)
basic_model_per_ticker_baseline_df = basic_model_per_ticker_results_df[
    basic_model_per_ticker_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["model_name", "ticker", "test_accuracy", "test_balanced_accuracy", "test_f1_score"]].rename(
    columns={
        "test_accuracy": "baseline_test_accuracy",
        "test_balanced_accuracy": "baseline_test_balanced_accuracy",
        "test_f1_score": "baseline_test_f1_score",
    }
)
basic_model_per_ticker_lift_df = basic_model_per_ticker_results_df.merge(
    basic_model_per_ticker_baseline_df,
    on=["model_name", "ticker"],
    how="left",
)
basic_model_per_ticker_lift_df["test_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_balanced_accuracy"]
    - basic_model_per_ticker_lift_df["baseline_test_balanced_accuracy"]
)
basic_model_per_ticker_lift_df["test_accuracy_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_accuracy"] - basic_model_per_ticker_lift_df["baseline_test_accuracy"]
)
basic_model_per_ticker_lift_df["test_f1_score_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_f1_score"] - basic_model_per_ticker_lift_df["baseline_test_f1_score"]
)

basic_model_ticker_best_alternative_df = (
    basic_model_per_ticker_lift_df[
        ~basic_model_per_ticker_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "ticker", "test_balanced_accuracy_lift_vs_price_volume", "test_f1_score_lift_vs_price_volume"],
        ascending=[True, True, False, False],
    )
    .groupby(["model_name", "ticker"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

basic_model_alternative_data_question_df


In [ ]:
from __future__ import annotations

basic_model_ticker_help_summary_df = (
    basic_model_ticker_best_alternative_df.assign(
        helps_test_balanced_accuracy=lambda frame: frame["test_balanced_accuracy_lift_vs_price_volume"] > 0
    )
    .groupby(["model_name", "model_family"])
    .agg(
        tickers_with_positive_alt_lift=("helps_test_balanced_accuracy", "sum"),
        tickers_evaluated=("ticker", "nunique"),
        mean_best_alt_balanced_accuracy_lift=("test_balanced_accuracy_lift_vs_price_volume", "mean"),
        median_best_alt_balanced_accuracy_lift=("test_balanced_accuracy_lift_vs_price_volume", "median"),
        best_single_ticker_lift=("test_balanced_accuracy_lift_vs_price_volume", "max"),
        worst_single_ticker_lift=("test_balanced_accuracy_lift_vs_price_volume", "min"),
    )
    .reset_index()
)
basic_model_ticker_help_summary_df["share_tickers_with_positive_alt_lift"] = (
    basic_model_ticker_help_summary_df["tickers_with_positive_alt_lift"]
    / basic_model_ticker_help_summary_df["tickers_evaluated"]
)

basic_model_ticker_attention_concern_df = (
    basic_model_ticker_best_alternative_df.merge(
        attention_coverage_by_ticker_df[
            ["ticker", "avg_attention_rank", "low_overall_attention_flag", "weakest_attention_source"]
        ],
        on="ticker",
        how="left",
    )
    .sort_values(["model_name", "low_overall_attention_flag", "test_balanced_accuracy_lift_vs_price_volume"], ascending=[True, False, True])
    .reset_index(drop=True)
)

basic_model_ticker_help_summary_df
